# Fenrir end-to-end demo

Este notebook enseña un flujo completo y utilizable de `Fenrir` sobre un dataset sintetico de clustering.
Importa primero la libreria instalada y, si no existe, usa el repo local sin depender de rutas absolutas.

Objetivos:
- ajustar el modelo y leer bien la mejor configuracion
- explotar leaderboard, metricas, perfiles de cluster y variables influyentes
- validar holdout, estabilidad y subconjuntos compactos
- exportar un workbook ejecutivo
- cerrar con una celda de validacion para comprobar que el flujo funciona correctamente

In [ ]:
from pathlib import Path
import importlib
import sys

import numpy as np
import pandas as pd
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler

candidate_project_roots = [
    Path.cwd(),
    Path.cwd() / "fenrir",
    Path.cwd().parent,
    Path.cwd().parent / "fenrir",
]

try:
    import fenrir as fenrir_package
except ModuleNotFoundError:
    project_root = next(
        (
            path
            for path in candidate_project_roots
            if (path / "core.py").exists() and (path / "__init__.py").exists()
        ),
        None,
    )
    if project_root is None:
        raise RuntimeError(
            "No se ha encontrado fenrir instalado ni una copia local del proyecto."
        )

    PACKAGE_PARENT = project_root.parent
    if str(PACKAGE_PARENT) not in sys.path:
        sys.path.insert(0, str(PACKAGE_PARENT))

    import fenrir as fenrir_package

fenrir_package = importlib.reload(fenrir_package)
Fenrir = fenrir_package.Fenrir
package_file = getattr(fenrir_package, "__file__", None)
if package_file is None:
    raise RuntimeError("No se pudo resolver la ruta del paquete fenrir cargado.")

PROJECT_ROOT = Path(package_file).resolve().parent

SEED = 13
print(f"Fenrir importado desde: {PROJECT_ROOT}")

Fenrir importado desde: C:\Users\alrodriguez\Downloads\fenrir


In [ ]:
blobs = make_blobs(
    n_samples=96,
    centers=3,
    n_features=4,
    cluster_std=0.65,
    random_state=SEED,
    return_centers=False,
)
values = blobs[0]
labels = blobs[1]

df = pd.DataFrame(values, columns=['x1', 'x2', 'x3', 'x4'])
df['segmento'] = pd.Series(labels).map({0: 'rojo', 1: 'azul', 2: 'verde'})
df['canal'] = np.where(labels == 0, 'web', np.where(labels == 1, 'store', 'partner'))
df['ticket_medio'] = np.where(labels == 0, 'alto', np.where(labels == 1, 'medio', 'bajo'))
df['target'] = labels

df.head()

,x1,x2,x3,x4,segmento,canal,ticket_medio,target
0,5.636795,-3.851427,6.879658,9.297582,rojo,web,alto,0
1,9.384930,-1.632373,2.607159,5.364489,azul,store,medio,1
2,8.333505,-0.919327,1.878312,5.770841,azul,store,medio,1
3,1.760053,4.619421,-8.838729,-4.669610,verde,partner,bajo,2
4,5.085398,-4.883705,6.280913,9.035551,rojo,web,alto,0


In [3]:
model = Fenrir(
    df,
    target='target',
    variance_threshold=0.85,
    cluster_range=range(3, 5),
    algorithms=('kmeans', 'gmm'),
    scalers={'StandardScaler': StandardScaler()},
    random_state=SEED,
)
labels_pred = model.fit_predict()

best_configuration = model.best_configuration().rename('valor').to_frame()
leaderboard = model.summary(top_n=5)
metrics = model.metric_report()

display(best_configuration)
display(leaderboard)
display(metrics)

,valor
scaler_name,StandardScaler
algorithm,kmeans
n_clusters,3
n_components,2
explained_variance,0.994114
variance_first_two,0.994114
silhouette,0.975457
calinski_harabasz,46437.602364
davies_bouldin,0.033526


,rank,scaler_name,algorithm,algorithm_label,configuration,n_clusters,n_components,explained_variance,explained_variance_reading,variance_first_two,variance_first_two_reading,silhouette,silhouette_reading,calinski_harabasz,davies_bouldin,davies_bouldin_reading
0,1,StandardScaler,kmeans,K-Means,K-Means | k=3 | PCA=2,3,2,0.9941,Muy fuerte,0.9941,Muy util para visualizacion 2D,0.9755,Separacion excelente,46437.6024,0.0335,Muy bueno
1,2,StandardScaler,gmm,Gaussian Mixture,Gaussian Mixture | k=3 | PCA=2,3,2,0.9941,Muy fuerte,0.9941,Muy util para visualizacion 2D,0.9755,Separacion excelente,46437.6024,0.0335,Muy bueno
2,3,StandardScaler,kmeans,K-Means,K-Means | k=4 | PCA=2,4,2,0.9941,Muy fuerte,0.9941,Muy util para visualizacion 2D,0.8284,Separacion excelente,37615.5102,0.3578,Muy bueno
3,4,StandardScaler,gmm,Gaussian Mixture,Gaussian Mixture | k=4 | PCA=2,4,2,0.9941,Muy fuerte,0.9941,Muy util para visualizacion 2D,0.7908,Separacion excelente,35551.5272,0.4579,Muy bueno


,metric,metric_label,value,value_display,direction,acceptable,interpretation,notes
0,explained_variance,Varianza explicada PCA,0.9941,99.4%,mas alto es mejor,0.70+ aceptable | 0.85+ fuerte | 0.90+ muy fuerte,Muy fuerte,No garantiza clusters buenos por si solo; mide...
1,variance_first_two,Varianza acumulada en PC1-PC2,0.9941,99.4%,mas alto es mejor,0.50+ suele permitir graficos 2D legibles,Muy util para visualizacion 2D,Sirve sobre todo para visualizacion; no sustit...
2,silhouette,Silhouette,0.9755,0.9755,mas alto es mejor,<0.25 flojo | 0.25-0.50 usable | 0.50-0.70 bue...,Separacion excelente,Es la referencia mas interpretable para separa...
3,davies_bouldin,Davies-Bouldin,0.0335,0.0335,mas bajo es mejor,<0.60 excelente | <1.00 bueno | 1.00-1.50 regu...,Muy bueno,"No tiene umbral universal, pero penaliza clust..."
4,calinski_harabasz,Calinski-Harabasz,46437.6024,46437.6024,mas alto es mejor,sin umbral absoluto,Solo interpretable en comparacion con otras co...,Se interpreta comparando configuraciones dentr...
5,ari,Adjusted Rand Index,1.0000,1.0000,mas alto es mejor,0.30+ senal | 0.50+ bueno | 0.75+ muy fuerte,Alineacion muy fuerte con la objetivo,Mide acuerdo con la variable objetivo externa ...
6,nmi,Normalized Mutual Information,1.0000,1.0000,mas alto es mejor,0.30+ senal | 0.50+ bueno | 0.75+ muy fuerte,Alineacion muy fuerte con la objetivo,Mide informacion compartida entre clusters y o...
7,v_measure,V-Measure,1.0000,1.0000,mas alto es mejor,0.30+ senal | 0.50+ bueno | 0.75+ muy fuerte,Alineacion muy fuerte con la objetivo,Equilibra homogeneidad y completitud respecto ...
8,n_clusters,Numero de clusters,3.0000,3,depende del problema,aceptable si es estable entre splits,Valido si tambien es estable en holdout y sigu...,Muchos clusters pequenos suelen ser fragiles a...
9,n_components,Componentes PCA,2.0000,2,menos es mejor si el rendimiento no cae,mantener solo las necesarias,"Mejor cuanto mas pequeno sea, si mantiene el r...","Sirve para balancear compresion, interpretabil..."


In [4]:
source_feature_report = model.source_feature_report(top_n=6)
influential_variables = model.list_influential_variables(top_n=6)
cluster_sizes = model.cluster_size_report()
cluster_profile = model.cluster_groupby(top_n=4)
cluster_vs_target = model.cluster_target_report(normalize='index')
influential_dataset = model.influential_dataset(top_n=4, include_target=True)

display(source_feature_report)
display(cluster_sizes)
display(cluster_profile)
display(cluster_vs_target)
display(influential_dataset.head())
influential_variables

,PC1_loading_absmax,PC2_loading_absmax,PC1_contribution,PC2_contribution,PC1_cos2_max,PC2_cos2_max,encoded_features,contribution_total,cos2_total,weighted_contribution,weighted_cos2
ticket_medio,0.991010,0.940891,19.040348,29.653521,0.971871,0.876053,3,48.693869,1.847925,23.190672,0.934401
canal,0.991010,0.940891,19.040348,29.653521,0.971871,0.876053,3,48.693869,1.847925,23.190672,0.934401
segmento,0.991010,0.940891,19.040348,29.653521,0.971871,0.876053,3,48.693869,1.847925,23.190672,0.934401
x3,0.996136,0.099917,12.477614,0.195486,0.981951,0.009879,1,12.673099,0.991831,7.674637,0.601819
x4,0.993904,0.118364,12.421745,0.274331,0.977555,0.013864,1,12.696077,0.991419,7.671450,0.600700
x2,0.956399,0.280826,11.501975,1.544220,0.905171,0.078042,1,13.046195,0.983213,7.607954,0.581719


,cluster_size,cluster_share,cluster_balance
cluster,,,
0,32,0.3333,Grande
1,32,0.3333,Grande
2,32,0.3333,Grande


,cluster_size,cluster_share,ticket_medio,canal,segmento,x3,target,cluster_balance
cluster,,,,,,,,
0,32,0.3333,bajo,partner,verde,-9.1892,2.0,Grande
1,32,0.3333,alto,web,rojo,6.3026,0.0,Grande
2,32,0.3333,medio,store,azul,2.1177,1.0,Grande


target,0,1,2,target_dominante,share_dominante,lectura_target
cluster,,,,,,
0,0.0,0.0,1.0,2,1.0,Muy concentrado
1,1.0,0.0,0.0,0,1.0,Muy concentrado
2,0.0,1.0,0.0,1,1.0,Muy concentrado


,ticket_medio,canal,segmento,x3,target
0,alto,web,rojo,6.879658,0
1,medio,store,azul,2.607159,1
2,medio,store,azul,1.878312,1
3,bajo,partner,verde,-8.838729,2
4,alto,web,rojo,6.280913,0


['ticket_medio', 'canal', 'segmento', 'x3', 'x4', 'x2']

In [5]:
holdout_summary = model.evaluate_holdout(test_size=0.25, n_splits=2)
holdout_top = model.holdout_report(top_n=3)
compact_search = model.search_influential_subsets(
    min_features=2,
    max_features=4,
    criterion='holdout_silhouette',
    include_holdout=True,
    test_size=0.25,
    n_splits=2,
)
best_compact_variables = model.best_influential_variable_list()
best_compact_model = model.best_influential_model()
stability_summary = model.evaluate_stability(sample_fraction=0.8, n_splits=2)
stability_top = model.stability_report(top_n=3)
analysis = model.analysis_report(
    top_n=5,
    leaderboard_top_n=5,
    holdout_top_n=3,
    compact_max_features=4,
    test_size=0.25,
    n_splits=2,
    include_stability=True,
    stability_n_splits=2,
    stability_top_n=3,
)

display(holdout_top)
display(compact_search.head(4))
display(stability_summary)
display(stability_top)
display(analysis['leaderboard'])
best_compact_variables

c:\Users\alrodriguez\Downloads\bahamut\.venv\Lib\site-packages\sklearn\base.py:1336: ConvergenceWarning: Number of distinct clusters (3) found smaller than n_clusters (4). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
c:\Users\alrodriguez\Downloads\bahamut\.venv\Lib\site-packages\sklearn\base.py:1336: ConvergenceWarning: Number of distinct clusters (3) found smaller than n_clusters (4). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
c:\Users\alrodriguez\Downloads\bahamut\.venv\Lib\site-packages\sklearn\base.py:1336: ConvergenceWarning: Number of distinct clusters (3) found smaller than n_clusters (4). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
c:\Users\alrodriguez\Downloads\bahamut\.venv\Lib\site-packages\sklearn\base.py:1336: ConvergenceWarning: Number of distinct clusters (3) found smaller than n_clusters (4). Possibly due to duplicate points in X.
  return fit

,rank,scaler_name,algorithm,algorithm_label,configuration,n_clusters,n_components,explained_variance_mean,explained_variance_reading,variance_first_two_mean,...,train_davies_bouldin_std,train_ari_mean,train_ari_std,test_ari_std,train_nmi_mean,train_nmi_std,test_nmi_std,train_v_measure_mean,train_v_measure_std,test_v_measure_std
0,1,StandardScaler,gmm,Gaussian Mixture,Gaussian Mixture | k=3 | PCA=2,3,2,0.9942,Muy fuerte,0.9942,...,0.000571,1.000000,0.000000,0.0000,1.000000,0.000000,0.0000,1.000000,0.000000,0.0000
1,2,StandardScaler,kmeans,K-Means,K-Means | k=3 | PCA=2,3,2,0.9942,Muy fuerte,0.9942,...,0.000571,1.000000,0.000000,0.0000,1.000000,0.000000,0.0000,1.000000,0.000000,0.0000
2,3,StandardScaler,kmeans,K-Means,K-Means | k=4 | PCA=2,4,2,0.9942,Muy fuerte,0.9942,...,0.008420,0.890507,0.013625,0.0522,0.916668,0.006686,0.0263,0.916668,0.006686,0.0263


,scaler_name,algorithm,n_clusters,n_components,explained_variance,variance_first_two,silhouette,calinski_harabasz,davies_bouldin,n_selected_features,...,test_davies_bouldin_std,test_v_measure_mean,test_nmi_mean,test_ari_mean,test_calinski_harabasz_mean,test_calinski_harabasz_std,test_ari_std,test_nmi_std,test_v_measure_std,selection_status
0,StandardScaler,kmeans,3,2,1.000000,1.000000,1.000000,1.403006e+32,7.024475e-09,2,...,0.0000,1.0,1.0,1.0,3.833375e+33,0.000000e+00,0.0,0.0,0.0,valido
1,StandardScaler,kmeans,3,2,1.000000,1.000000,1.000000,2.546457e+34,0.000000e+00,3,...,0.0000,1.0,1.0,1.0,2.332503e+65,3.298657e+65,0.0,0.0,0.0,valido
2,StandardScaler,kmeans,3,2,0.999362,0.999362,0.991848,3.293382e+05,1.144272e-02,4,...,0.0004,1.0,1.0,1.0,1.156652e+05,8.357147e+03,0.0,0.0,0.0,valido


,n_splits,sample_fraction,ari_vs_base_mean,ari_vs_base_std,nmi_vs_base_mean,nmi_vs_base_std,v_measure_vs_base_mean,v_measure_vs_base_std,same_scaler_rate,same_algorithm_rate,same_n_clusters_rate,same_n_components_rate,same_configuration_rate,silhouette_mean,min_cluster_size_mean,stability_reading
0,2,0.8,1.0,0.0,1.0,0.0,1.0,0.0,1.0,1.0,1.0,1.0,1.0,0.976286,25.0,Muy estable


,split,sample_size,sample_fraction,scaler_name,algorithm,algorithm_label,configuration,n_clusters,n_components,silhouette,...,v_measure_vs_base,min_cluster_size,same_configuration,configuration_status,explained_variance,davies_bouldin,same_scaler,same_algorithm,same_n_clusters,same_n_components
0,2,76,0.7917,StandardScaler,kmeans,K-Means,K-Means | k=3 | PCA=2,3,2,0.9764,...,1.0,25,True,Coincide con la configuracion base,0.9942,0.0316,True,True,True,True
1,1,76,0.7917,StandardScaler,kmeans,K-Means,K-Means | k=3 | PCA=2,3,2,0.9761,...,1.0,25,True,Coincide con la configuracion base,0.9943,0.0326,True,True,True,True


,rank,scaler_name,algorithm,algorithm_label,configuration,n_clusters,n_components,explained_variance,explained_variance_reading,variance_first_two,variance_first_two_reading,silhouette,silhouette_reading,calinski_harabasz,davies_bouldin,davies_bouldin_reading
0,1,StandardScaler,kmeans,K-Means,K-Means | k=3 | PCA=2,3,2,0.9941,Muy fuerte,0.9941,Muy util para visualizacion 2D,0.9755,Separacion excelente,46437.6024,0.0335,Muy bueno
1,2,StandardScaler,gmm,Gaussian Mixture,Gaussian Mixture | k=3 | PCA=2,3,2,0.9941,Muy fuerte,0.9941,Muy util para visualizacion 2D,0.9755,Separacion excelente,46437.6024,0.0335,Muy bueno
2,3,StandardScaler,kmeans,K-Means,K-Means | k=4 | PCA=2,4,2,0.9941,Muy fuerte,0.9941,Muy util para visualizacion 2D,0.8284,Separacion excelente,37615.5102,0.3578,Muy bueno
3,4,StandardScaler,gmm,Gaussian Mixture,Gaussian Mixture | k=4 | PCA=2,4,2,0.9941,Muy fuerte,0.9941,Muy util para visualizacion 2D,0.7908,Separacion excelente,35551.5272,0.4579,Muy bueno


['ticket_medio', 'canal']

In [6]:
export_dir = PROJECT_ROOT / 'examples' / 'exports'
export_dir.mkdir(parents=True, exist_ok=True)
export_path = model.export_executive_workbook(file_name=str(export_dir / 'fenrir_demo.xlsx'))
print(export_path)
Path(export_path).exists()

c:\Users\alrodriguez\Downloads\bahamut\.venv\Lib\site-packages\sklearn\base.py:1336: ConvergenceWarning: Number of distinct clusters (3) found smaller than n_clusters (4). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
c:\Users\alrodriguez\Downloads\bahamut\.venv\Lib\site-packages\sklearn\base.py:1336: ConvergenceWarning: Number of distinct clusters (3) found smaller than n_clusters (4). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
c:\Users\alrodriguez\Downloads\bahamut\.venv\Lib\site-packages\sklearn\base.py:1336: ConvergenceWarning: Number of distinct clusters (3) found smaller than n_clusters (4). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)
c:\Users\alrodriguez\Downloads\bahamut\.venv\Lib\site-packages\sklearn\base.py:1336: ConvergenceWarning: Number of distinct clusters (3) found smaller than n_clusters (4). Possibly due to duplicate points in X.
  return fit

c:\Users\alrodriguez\Downloads\fenrir\examples\exports\fenrir_demo.xlsx


True

In [7]:
assert len(labels_pred) == len(df)
assert hasattr(model, 'interactive_workbench')
assert not leaderboard.empty and 'configuration' in leaderboard.columns
assert {'metric_label', 'value_display'}.issubset(metrics.columns)
assert {'cluster_size', 'cluster_share', 'cluster_balance'}.issubset(cluster_sizes.columns)
assert {'target_dominante', 'share_dominante', 'lectura_target'}.issubset(cluster_vs_target.columns)
assert not holdout_top.empty and 'configuration' in holdout_top.columns
assert not stability_top.empty and 'configuration_status' in stability_top.columns
assert isinstance(best_compact_variables, list) and len(best_compact_variables) >= 2
assert best_compact_model.best_configuration().shape[0] > 0
assert 'leaderboard' in analysis and not analysis['leaderboard'].empty
assert Path(export_path).exists()
print('Notebook Fenrir validado correctamente.')

Notebook Fenrir validado correctamente.
